# Exercise 1: FST architecture

*Finite-State Morphology · version 0.1.0*

> Draw a state diagram for a transducer that analyses Turkish nouns with singular/plural number and nominative/dative case: four cells per root (*ev*, *ev-ler*, *ev-e*, *ev-ler-e*), one of which is the bare root. Fix your assumptions first (alphabet, whether tags are separate symbols, one stem or a lexicon of stems). Under those assumptions, how many states do you need, and which transitions? How does the diagram change if the suffixes contain the archiphoneme `{A}` of the toy grammar?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0"`. These notebooks compile lexc and twolc grammars, which needs the compiled `hfst` package; on PyPI it had wheels for Python 3.11 but not for 3.13 when checked on 19 October 2026, so use Python 3.11 or 3.12 if `pip` cannot find it. The Apertium transducers are downloaded once with `turkicnlp.download`.

**Assumptions.** Letters are single symbols; each tag (`<n>`, `<pl>`, `<dat>`) is one multicharacter symbol; one stem, *ev*; the analysis side carries the tags and the surface side the letters, as in the chapter's toy grammar. Under these assumptions, HFST can build the minimal transducer from a three-lexicon lexc file, and its states and arcs answer the question. The second lexicon adds *araba* and writes the suffix vowel as `{A}`, to show what changes.

In [ ]:
import tempfile
from pathlib import Path
import hfst
import turkicnlp
from turkicnlp.resources.registry import ModelRegistry

def compile_grammar(lexc_text, twolc_text=None):
    """Compile a lexc lexicon (and optional twolc rules); return (analyser, generator) in lookup format."""
    work = Path(tempfile.mkdtemp())
    (work / "g.lexc").write_text(lexc_text, encoding="utf-8")
    gen = hfst.compile_lexc_file(str(work / "g.lexc"))
    if twolc_text:
        (work / "g.twolc").write_text(twolc_text, encoding="utf-8")
        hfst.compile_twolc_file(str(work / "g.twolc"), str(work / "g.twol.hfst"))
        stream, rules = hfst.HfstInputStream(str(work / "g.twol.hfst")), []
        while not stream.is_eof():
            rules.append(stream.read())
        stream.close()
        gen.compose_intersect(tuple(rules))
    gen.minimize()
    ana = hfst.HfstTransducer(gen)
    ana.invert()
    ana.minimize()
    for t in (ana, gen):
        t.convert(hfst.ImplementationType.HFST_OLW_TYPE)
    return ana, gen

def apertium(lang, script, kind="automorf"):
    """An Apertium transducer distributed with TurkicNLP (downloaded on first use)."""
    turkicnlp.download(lang, processors=["morph"])
    path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.{kind}.hfst"
    stream = hfst.HfstInputStream(str(path))
    fst = stream.read()
    stream.close()
    return fst

lookup = lambda fst, s: sorted({out for out, _ in fst.lookup(s)})

In [ ]:
EXPLICIT = """Multichar_Symbols
%<n%> %<pl%> %<dat%>
LEXICON Root
ev%<n%>:ev Number ;
LEXICON Number
%<pl%>:ler Case ;
           Case ;
LEXICON Case
           # ;
%<dat%>:e  # ;
"""
ARCHIPHONEME = """Multichar_Symbols
%<n%> %<pl%> %<dat%> %{A%}
LEXICON Root
ev%<n%>:ev       Number ;
araba%<n%>:araba Number ;
LEXICON Number
%<pl%>:l%{A%}r Case ;
           Case ;
LEXICON Case
           # ;
%<dat%>:%{A%}  # ;
"""

def describe(name, lexc_text):
    fst = hfst.compile_lexc_file(_write(lexc_text))
    fst.minimize()
    basic = hfst.HfstBasicTransducer(fst)
    states = list(basic.states())
    arcs = [(s, t.get_target_state(), t.get_input_symbol(), t.get_output_symbol()) for s in states for t in basic.transitions(s)]
    finals = [s for s in states if basic.is_final_state(s)]
    print(f"{name}: {len(states)} states, {len(arcs)} arcs, final states {finals}")
    for s, d, i, o in arcs:
        eps = "@_EPSILON_SYMBOL_@"
        print(f"   {s} -> {d}   {i.replace(eps, '0')} : {o.replace(eps, '0')}")

def _write(text):
    p = Path(tempfile.mkdtemp()) / "g.lexc"
    p.write_text(text, encoding="utf-8")
    return str(p)

describe("one stem, explicit allomorphs", EXPLICIT)
describe("two stems, archiphoneme {A} (before rules)", ARCHIPHONEME)
ana, gen = compile_grammar(EXPLICIT)
print({w: lookup(ana, w) for w in ["ev", "evler", "eve", "evlere", "evlerde"]})

## Worked solution

Output of `experiments/ch06_exercises/run_ch06_exercises.py` on 19 October 2026 (TurkicNLP 0.5.0, HFST 3.16.0.1, Python 3.11):

```
one stem, explicit allomorphs: 8 states, 8 arcs, final states [3, 5, 7]
   0 -> 1   e : e
   1 -> 2   v : v
   2 -> 3   <n> : 0
   3 -> 4   <pl> : l
   3 -> 5   <dat> : e
   4 -> 6   0 : e
   6 -> 7   0 : r
   7 -> 5   <dat> : e
two stems, archiphoneme {A} (before rules): 12 states, 13 arcs, final states [6, 9, 11]
   0 -> 1   a : a
   0 -> 2   e : e
   1 -> 3   r : r
   2 -> 4   v : v
   3 -> 5   a : a
   4 -> 6   <n> : 0
   5 -> 7   b : b
   6 -> 8   <pl> : l
   6 -> 9   <dat> : {A}
   7 -> 4   a : a
   8 -> 10   0 : {A}
   10 -> 11   0 : r
   11 -> 9   <dat> : {A}
{'ev': ['ev<n>'], 'evler': ['ev<n><pl>'], 'eve': ['ev<n><dat>'], 'evlere': ['ev<n><pl><dat>'], 'evlerde': []}
```

**One stem.** The minimal transducer has 8 states and 8 arcs, with 3 final states (state numbers as printed above, `0` = empty string):

- `0 –e:e→ 1 –v:v→ 2 –<n>:0→ 3`: the stem; state 3 is final, the bare root *ev* (singular nominative). Singular and nominative are empty suffixes, so they need no arcs of their own.
- `3 –<dat>:e→ 5`: *eve*; state 5 is final.
- `3 –<pl>:l→ 4 –0:e→ 6 –0:r→ 7`: the plural, one arc per letter with the tag on the first; state 7 is final, *evler*.
- `7 –<dat>:e→ 5`: *evlere*. The two dative arcs end in the same final state, which minimisation merges.

**With `{A}` and a second stem.** The lexicon level has 12 states and 13 arcs. The stems need their own letter paths (minimisation joins *araba* to the path of *ev* after its last letter, state 4, because both continue the same way), but the suffix part keeps exactly the same shape: one plural path and one dative arc, now with `{A}` on the surface side. Without the archiphoneme, a lexicon with front and back stems would need separate suffix paths, or separate continuation classes, for *-ler/-e* and *-lar/-a*. With it, the choice of *a* or *e* moves into the harmony rule, which is composed with this transducer afterwards and adds its own states.

## Your turn — extension

1. Add the locative *-de* and count again. Which states are reused?
2. Compose the archiphoneme lexicon with the chapter's harmony rule (the toy grammar's twolc file) and count the states of the result.